# Real-Time Market Data Streaming

This notebook demonstrates how to use the CBSC Strategy SDK's real-time WebSocket client for live market data streaming in Jupyter notebooks.

## Features

- WebSocket client with auto-reconnection
- Async iterator pattern for data consumption
- Callback-based event system
- Circular buffers for data aggregation
- Live Plotly chart updates

## Requirements

```bash
pip install websockets plotly pandas
```

## 1. Basic Setup

In [ ]:
# Import required modules
import asyncio
import nest_asyncio
from datetime import datetime

# Apply nest_asyncio for Jupyter compatibility
nest_asyncio.apply()

# Import SDK components
from cbsc_strategy_sdk.data import (
    RealTimeDataStream,
    EventEmitter,
    CircularBuffer,
    TickCircularBuffer,
    TickData,
)
from cbsc_strategy_sdk.visualization import LiveChart, LiveChartManager

print("✅ All imports successful!")

## 2. Event System Demo

The EventEmitter provides a callback-based event system for real-time updates.

In [ ]:
# Create event emitter
emitter = EventEmitter()

# Register callbacks
tick_count = [0]

def on_tick(tick):
    tick_count[0] += 1
    print(f"📊 Tick #{tick_count[0]}: {tick}")

def on_error(error):
    print(f"❌ Error: {error}")

emitter.on('tick', on_tick)
emitter.on('error', on_error)

# Emit events
await emitter.emit('tick', TickData(
    symbol='AAPL',
    timestamp=datetime.now(),
    price=150.25,
    volume=1000,
))

await emitter.emit('tick', TickData(
    symbol='AAPL',
    timestamp=datetime.now(),
    price=150.50,
    volume=2000,
))

## 3. Circular Buffer Demo

Circular buffers automatically manage memory by discarding old data when full.

In [ ]:
# Create a circular buffer with max 10 items
buffer = TickCircularBuffer(size=10, symbol='AAPL')

# Add some ticks
for i in range(15):
    buffer.append(TickData(
        symbol='AAPL',
        timestamp=datetime.now(),
        price=150.0 + i * 0.1,
        volume=1000 + i * 100,
    ))

print(f"Buffer size: {buffer.current_size} (max: {buffer.size})")
print(f"Total appended: {buffer.total_appended}")
print(f"Latest price: {buffer.get_latest_price()}")

# Convert to DataFrame
df = buffer.to_dataframe()
print(f"\nDataFrame shape: {df.shape}")
df.head()

## 4. OHLCV Aggregation

Aggregate tick data into OHLCV bars for analysis.

In [ ]:
# Convert ticks to OHLCV bars
ohlcv = buffer.to_ohlcv(interval='1s')

print(f"OHLCV bars: {len(ohlcv)}")
print("\nFirst few bars:")
ohlcv.head()

## 5. Live Chart Demo

Create real-time updating charts with Plotly.

In [ ]:
# Create a live chart
chart = LiveChart(
    title="AAPL Live Price",
    max_points=50,
)

# Add a line trace
chart.add_line(name='price', color='blue')

# Simulate real-time updates
for i in range(20):
    tick = TickData(
        symbol='AAPL',
        timestamp=datetime.now(),
        price=150.0 + i * 0.05,
        volume=1000,
    )
    chart.update(tick)

# Display the chart
chart.show()

## 6. Live Chart Manager

Manage multiple live charts for different symbols.

In [ ]:
# Create chart manager
manager = LiveChartManager()

# Add charts for multiple symbols
manager.add_symbol('AAPL', 'Apple Live Price', max_points=30)
manager.add_symbol('0700.HK', 'Tencent Live Price', max_points=30)

# Simulate updates
for i in range(15):
    # AAPL tick
    aapl_tick = TickData(
        symbol='AAPL',
        timestamp=datetime.now(),
        price=150.0 + i * 0.1,
        volume=1000,
    )
    manager.update(aapl_tick)
    
    # Tencent tick
    tencent_tick = TickData(
        symbol='0700.HK',
        timestamp=datetime.now(),
        price=350.0 + i * 0.5,
        volume=5000,
    )
    manager.update(tencent_tick)

# Show all charts
manager.show_all()

## 7. Real-Time WebSocket Stream (Simulation)

Simulate a WebSocket connection with mock data.

**Note**: For actual WebSocket connections, you need:
- Running CBSC WebSocket server at `ws://localhost:3007/ws/realtime`
- Valid JWT authentication token

In [ ]:
async def simulate_websocket_stream():
    """Simulate WebSocket stream for demonstration."""
    
    # Create stream (without actual connection)
    stream = RealTimeDataStream(
        ws_url='ws://localhost:3007/ws/realtime',
        auth_token='mock-token',
        symbols=['AAPL'],
        buffer_size=50,
    )
    
    # Create live chart
    chart = LiveChart(title="Live AAPL Price", max_points=50)
    chart.add_line(name='price', color='green')
    
    print("🔌 Simulating WebSocket connection...")
    
    # Simulate receiving ticks
    base_price = 150.0
    for i in range(20):
        # Simulate price movement
        price_change = (i % 5 - 2) * 0.05  # Random-ish movement
        tick = TickData(
            symbol='AAPL',
            timestamp=datetime.now(),
            price=base_price + price_change,
            volume=1000 + i * 100,
        )
        
        # Update buffer
        stream._buffers['AAPL'].append(tick)
        
        # Update chart
        chart.update(tick)
        
        # Simulate network delay
        await asyncio.sleep(0.1)
    
    print("✅ Simulation complete!")
    
    # Show results
    buffer = stream.get_buffer('AAPL')
    print(f"\n📊 Buffer contains {buffer.current_size} ticks")
    print(f"Latest price: {buffer.get_latest_price()}")
    
    # Show chart
    chart.show()

# Run simulation
await simulate_websocket_stream()

## 8. Actual WebSocket Connection Example

Example code for connecting to real WebSocket server (uncomment to use with actual server):

In [ ]:
'''
async def connect_to_real_websocket():
    """Connect to actual CBSC WebSocket server."""
    
    # Create stream with real configuration
    stream = RealTimeDataStream(
        ws_url='ws://localhost:3007/ws/realtime',
        auth_token='your-jwt-token-here',  # Get from StrategyWorkspace
        symbols=['AAPL', '0700.HK'],
        buffer_size=1000,
        max_retries=5,
    )
    
    # Set up event callbacks
    @stream.on('connected')
    def on_connected():
        print("✅ Connected to WebSocket!")
    
    @stream.on('tick')
    def on_tick(tick):
        print(f"📊 Received: {tick}")
    
    @stream.on('error')
    def on_error(error):
        print(f"❌ Error: {error}")
    
    @stream.on('disconnected')
    def on_disconnected():
        print("🔌 Disconnected from WebSocket")
    
    try:
        # Connect to server
        await stream.connect()
        
        # Stream ticks for 30 seconds
        start_time = datetime.now()
        async for tick in stream.stream():
            # Process tick
            elapsed = (datetime.now() - start_time).total_seconds()
            if elapsed > 30:
                break
    
    finally:
        # Always disconnect
        await stream.disconnect()

# Uncomment to run with real server:
# await connect_to_real_websocket()
'''

## 9. Best Practices

### Memory Management
- Use appropriate buffer sizes to balance memory and data retention
- Call `buffer.clear()` when done with data
- Use `latest(n)` to get only the data you need

### Error Handling
- Always handle `disconnected` events
- Use try-finally to ensure cleanup
- Monitor connection status with `is_connected`

### Performance
- Use async iteration for efficient streaming
- Batch operations when possible
- Consider aggregation for high-frequency data

### Chart Updates
- Use `max_points` to limit chart data
- Update charts from buffers, not individual ticks
- Use FigureWidget for smooth Jupyter updates

## Summary

This notebook demonstrated:

✅ EventEmitter for callback-based events  
✅ CircularBuffer for efficient data storage  
✅ LiveChart for real-time visualization  
✅ RealTimeDataStream for WebSocket connections  
✅ OHLCV aggregation from tick data  
✅ Multi-symbol chart management  

### Next Steps

- Connect to real CBSC WebSocket server
- Implement custom trading strategies
- Add technical indicators on live data
- Build custom dashboards with multiple charts